# R7 · Limites et dérivation — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r7_limites_derivation/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r7_limites_derivation/03_notebook.ipynb`, créée par `python tools/start_chapter.py R7`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R7.1 à R7.11, R7.15 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R7.17 | Le taux d'accroissement · `difference_quotient` | 🔨 | ★ | 15 |
|  | R7.18 | La tangente à partir de f et de sa dérivée · `tangent_line` | 🔨 | ★★ | 20 |
|  | R7.19 | Trouver un zéro par dichotomie · `bisection` | 🔨 | ★★ | 25 |
| B | R7.20 | Vérifier ses dérivées avec `sympy.diff` | 📦 | ★ | 15 |
|  | R7.21 | Prédire : le taux d'accroissement a-t-il toujours une limite quand h tend vers 0 ? | 🔮 | ★★ | 15 |
|  | R7.22 | Zoomer sur une courbe jusqu'à voir une droite | 🔬 | ★★ | 20 |
|  | R7.23 | Tracer f, f′ et des tangentes | 🎨 | ★★ | 20 |
| C | R7.24 | Chasse à l'erreur : quatre dérivées fausses | 🐛 | ★★ | 20 |
|  | R7.25 | Tester une dérivée avec pytest et SymPy | 🛠️ | ★ | 15 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R7")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : limites et asymptotes, taux d'accroissement, tangentes, dérivées (usuelles, produits, quotients, composées, une lettre à la fois), variations et optimisation.
- Écrire les trois fonctions de `mymath/deriv.py` et les tester.
- Dériver avec SymPy, prévoir quand un taux d'accroissement a une limite, zoomer sur une courbe, tracer f et f′, chasser des dérivées fausses et écrire un test.

**Rappel express.** $f'(a) = \lim_{h \to 0} \frac{f(a + h) - f(a)}{h}$ ; tangente : $y = f(a) + f'(a)(x - a)$. $(x^n)' = n x^{n-1}$, $(e^x)' = e^x$, $(\ln x)' = \frac{1}{x}$, $(\sqrt{x})' = \frac{1}{2\sqrt{x}}$. $(uv)' = u'v + uv'$, $\left(\frac{u}{v}\right)' = \frac{u'v - uv'}{v^2}$, $\big(g(u)\big)' = g'(u) \times u'$. $f' > 0$ : $f$ croît ; un extremum intérieur annule $f'$.

## Partie 0 · Vérifier tes exercices papier (✏️ R7.1 à R7.11, 🔗 R7.15)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un nombre avec un point décimal, `-1.5`, arrondi comme le demande la question ; une limite infinie par `"+oo"` ou `"-oo"` ;
- une fraction exacte entre guillemets, `"5/7"` ;
- une expression entre guillemets, `"3*x**2 - 1"` (`*` pour multiplier, `**` pour la puissance ; `exp(x)`, `log(x)` pour ln x, `sqrt(x)`, `pi`) ;
- un intervalle entre guillemets, `"[-2 ; 4["` (`-oo` et `+oo` pour l'infini) ;
- une liste de nombres entre crochets (une tangente $y = mx + p$ par `[m, p]`) ; vrai ou faux par `True` ou `False`.

Les réponses pas encore remplies affichent ⏳.

**Ex R7.1 — Limites en l'infini des fonctions de référence**

In [ ]:
# Ex R7.1: the values you found on paper
answer_R7_1a = ...  # a) a number
answer_R7_1b = ...  # b) a number, or "+oo" / "-oo"
answer_R7_1c = ...  # c) a number
answer_R7_1d = ...  # d) a number
answer_R7_1e = ...  # e) a number
answer_R7_1f = ...  # f) a number
answer_R7_1g = ...  # g) a number, or "+oo" / "-oo"
answer_R7_1h = ...  # h) a number, or "+oo" / "-oo"

for letter, answer in zip("abcdefgh", [answer_R7_1a, answer_R7_1b, answer_R7_1c, answer_R7_1d, answer_R7_1e, answer_R7_1f, answer_R7_1g, answer_R7_1h]):
    wbm.check(f"R7.1{letter}", answer)

**Ex R7.2 — Limites et asymptotes de fonctions rationnelles**

In [ ]:
# Ex R7.2: the values you found on paper
answer_R7_2a = ...  # a) a number
answer_R7_2b = ...  # b) a number, or "+oo" / "-oo"
answer_R7_2c = ...  # c) a number, or "+oo" / "-oo"
answer_R7_2d = ...  # d) a list [a, L]
answer_R7_2e = ...  # e) a number
answer_R7_2f = ...  # f) True or False
answer_R7_2g = ...  # g) a list (increasing order)
answer_R7_2h = ...  # h) a number, or "+oo" / "-oo"

for letter, answer in zip("abcdefgh", [answer_R7_2a, answer_R7_2b, answer_R7_2c, answer_R7_2d, answer_R7_2e, answer_R7_2f, answer_R7_2g, answer_R7_2h]):
    wbm.check(f"R7.2{letter}", answer)

**Ex R7.3 — Taux d'accroissement et nombre dérivé**

In [ ]:
# Ex R7.3: the values you found on paper
answer_R7_3a = ...  # a) a number
answer_R7_3b = ...  # b) a number
answer_R7_3c = ...  # c) an expression in h, as text
answer_R7_3d = ...  # d) a number
answer_R7_3e = ...  # e) a number
answer_R7_3f = ...  # f) a number
answer_R7_3g = ...  # g) a number
answer_R7_3h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_R7_3a, answer_R7_3b, answer_R7_3c, answer_R7_3d, answer_R7_3e, answer_R7_3f, answer_R7_3g, answer_R7_3h]):
    wbm.check(f"R7.3{letter}", answer)

**Ex R7.4 — Équation de la tangente**

In [ ]:
# Ex R7.4: the values you found on paper
answer_R7_4a = ...  # a) a list [m, p]
answer_R7_4b = ...  # b) a list [m, p] (3 decimals)
answer_R7_4c = ...  # c) a list [m, p] (3 decimals)
answer_R7_4d = ...  # d) a list [m, p]
answer_R7_4e = ...  # e) a number (2 decimals)
answer_R7_4f = ...  # f) a number (5 decimals)
answer_R7_4g = ...  # g) a number
answer_R7_4h = ...  # h) a fraction, as text

for letter, answer in zip("abcdefgh", [answer_R7_4a, answer_R7_4b, answer_R7_4c, answer_R7_4d, answer_R7_4e, answer_R7_4f, answer_R7_4g, answer_R7_4h]):
    wbm.check(f"R7.4{letter}", answer)

**Ex R7.5 — Dériver les fonctions usuelles**

In [ ]:
# Ex R7.5: the values you found on paper
answer_R7_5a = ...  # a) an expression in x, as text
answer_R7_5b = ...  # b) an expression in x, as text
answer_R7_5c = ...  # c) an expression in x, as text
answer_R7_5d = ...  # d) an expression in x, as text
answer_R7_5e = ...  # e) an expression in x, as text
answer_R7_5f = ...  # f) an expression in x, as text
answer_R7_5g = ...  # g) an expression in x, as text
answer_R7_5h = ...  # h) an expression in x, as text

for letter, answer in zip("abcdefgh", [answer_R7_5a, answer_R7_5b, answer_R7_5c, answer_R7_5d, answer_R7_5e, answer_R7_5f, answer_R7_5g, answer_R7_5h]):
    wbm.check(f"R7.5{letter}", answer)

**Ex R7.6 — Dériver des produits et des quotients**

In [ ]:
# Ex R7.6: the values you found on paper
answer_R7_6a = ...  # a) an expression in x, as text
answer_R7_6b = ...  # b) an expression in x, as text
answer_R7_6c = ...  # c) an expression in x, as text
answer_R7_6d = ...  # d) an expression in x, as text
answer_R7_6e = ...  # e) an expression in x, as text
answer_R7_6f = ...  # f) an expression in x, as text
answer_R7_6g = ...  # g) a number (3 decimals)
answer_R7_6h = ...  # h) an expression in x, as text

for letter, answer in zip("abcdefgh", [answer_R7_6a, answer_R7_6b, answer_R7_6c, answer_R7_6d, answer_R7_6e, answer_R7_6f, answer_R7_6g, answer_R7_6h]):
    wbm.check(f"R7.6{letter}", answer)

**Ex R7.7 — Dériver des composées**

In [ ]:
# Ex R7.7: the values you found on paper
answer_R7_7a = ...  # a) an expression in x, as text
answer_R7_7b = ...  # b) an expression in x, as text
answer_R7_7c = ...  # c) an expression in x, as text
answer_R7_7d = ...  # d) an expression in x, as text
answer_R7_7e = ...  # e) an expression in x, as text
answer_R7_7f = ...  # f) an expression in x, as text
answer_R7_7g = ...  # g) an expression in x, as text
answer_R7_7h = ...  # h) a fraction, as text

for letter, answer in zip("abcdefgh", [answer_R7_7a, answer_R7_7b, answer_R7_7c, answer_R7_7d, answer_R7_7e, answer_R7_7f, answer_R7_7g, answer_R7_7h]):
    wbm.check(f"R7.7{letter}", answer)

**Ex R7.8 — Variations et tableau de variations**

In [ ]:
# Ex R7.8: the values you found on paper
answer_R7_8a = ...  # a) an expression in x, as text
answer_R7_8b = ...  # b) a list (increasing order)
answer_R7_8c = ...  # c) an interval, as text
answer_R7_8d = ...  # d) an integer
answer_R7_8e = ...  # e) a number
answer_R7_8f = ...  # f) a number
answer_R7_8g = ...  # g) a number
answer_R7_8h = ...  # h) a number (3 decimals)

for letter, answer in zip("abcdefgh", [answer_R7_8a, answer_R7_8b, answer_R7_8c, answer_R7_8d, answer_R7_8e, answer_R7_8f, answer_R7_8g, answer_R7_8h]):
    wbm.check(f"R7.8{letter}", answer)

**Ex R7.9 — Optimiser : la boîte de volume maximal**

In [ ]:
# Ex R7.9: the values you found on paper
answer_R7_9a = ...  # a) an interval, as text
answer_R7_9b = ...  # b) an expression in x, as text
answer_R7_9c = ...  # c) an expression in x, as text
answer_R7_9d = ...  # d) a number
answer_R7_9e = ...  # e) a number
answer_R7_9f = ...  # f) a list

for letter, answer in zip("abcdef", [answer_R7_9a, answer_R7_9b, answer_R7_9c, answer_R7_9d, answer_R7_9e, answer_R7_9f]):
    wbm.check(f"R7.9{letter}", answer)

**Ex R7.10 — La règle de la chaîne en série : trois étages, et l'ordre qui compte**

In [ ]:
# Ex R7.10: the values you found on paper
answer_R7_10a = ...  # a) an expression in x, as text
answer_R7_10b = ...  # b) an expression in x, as text
answer_R7_10c = ...  # c) an expression in x, as text
answer_R7_10d = ...  # d) an expression in x, as text
answer_R7_10e = ...  # e) a number
answer_R7_10f = ...  # f) a number (3 decimals)
answer_R7_10g = ...  # g) a number
answer_R7_10h = ...  # h) a list (3 decimals)

for letter, answer in zip("abcdefgh", [answer_R7_10a, answer_R7_10b, answer_R7_10c, answer_R7_10d, answer_R7_10e, answer_R7_10f, answer_R7_10g, answer_R7_10h]):
    wbm.check(f"R7.10{letter}", answer)

**Ex R7.11 — Dériver par rapport à une lettre, les autres fixées : A e^(kt), π r² h, ½ m v²**

In [ ]:
# Ex R7.11: the values you found on paper
answer_R7_11a = ...  # a) an expression in A, k, t, as text
answer_R7_11b = ...  # b) an expression in A, k, t, as text
answer_R7_11c = ...  # c) an expression in A, k, t, as text
answer_R7_11d = ...  # d) an expression in r, h, as text
answer_R7_11e = ...  # e) an expression in r, h, as text
answer_R7_11f = ...  # f) a number (2 decimals)
answer_R7_11g = ...  # g) an expression in m, v, as text
answer_R7_11h = ...  # h) an expression in m, v, as text

for letter, answer in zip("abcdefgh", [answer_R7_11a, answer_R7_11b, answer_R7_11c, answer_R7_11d, answer_R7_11e, answer_R7_11f, answer_R7_11g, answer_R7_11h]):
    wbm.check(f"R7.11{letter}", answer)

**Ex R7.15 — Minimiser une fonction de coût donnée par une formule : dériver, annuler, vérifier**

In [ ]:
# Ex R7.15: the values you found on paper
answer_R7_15a = ...  # a) an expression in a, as text
answer_R7_15b = ...  # b) a number
answer_R7_15c = ...  # c) a number
answer_R7_15d = ...  # d) a number
answer_R7_15e = ...  # e) a number (3 decimals)
answer_R7_15f = ...  # f) a number
answer_R7_15g = ...  # g) a list (3 decimals)

for letter, answer in zip("abcdefg", [answer_R7_15a, answer_R7_15b, answer_R7_15c, answer_R7_15d, answer_R7_15e, answer_R7_15f, answer_R7_15g]):
    wbm.check(f"R7.15{letter}", answer)

## Partie A · Ta librairie : taux d'accroissement, tangente, dichotomie

Fiche §2 à §4. Les trois fonctions de `mymath/deriv.py`.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from wbm.nbtools import checked, drawn, run_pytest_group


def counted(f, limit=10_000):
    """f, but a loop that calls it more than `limit` times stops with an error (a bisection needs far fewer)."""
    calls = 0

    def wrapped(x):
        nonlocal calls
        calls += 1
        if calls > limit:
            raise RuntimeError(f"f appelée plus de {limit} fois : ta boucle s'arrête-t-elle ? (tol, max_iter)")
        return f(x)

    return wrapped


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex R7.17 — Le taux d'accroissement · `difference_quotient` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le taux d'accroissement d'une fonction reçue en argument, et s'en servir pour estimer une dérivée.  
**Prérequis :** fiche §3 · **mymath :** `deriv.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme dans les chapitres précédents). Ouvre ta copie `mon_travail/mymath/deriv.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Comme dans tous les chapitres R0 à R9, Python pur et le module `math` seulement (un test le vérifie) : `f` est une fonction que tu reçois en argument, et tu l'appelles comme n'importe quelle fonction (workbookIA 0A §100.5.3). Les autres tests comparent tes résultats à des valeurs exactes, à SymPy et à SciPy.

Écris `difference_quotient(f, a, h)` dans `mon_travail/mymath/deriv.py` : le taux d'accroissement $\frac{f(a + h) - f(a)}{h}$ de `f` entre `a` et `a + h`, en flottant. Un `h` négatif est permis ; un `h` nul lève une `ValueError`.

La vérification calcule :
- a) les taux de $\sqrt{x}$ entre 4 et $4 + h$, pour $h = 1$, puis 0,1, puis 0,01 (une liste) ;
- b) le même taux pour $h = -0{,}01$ ;
- c) le nom de l'erreur levée pour $h = 0$ (avec `raised`, de la cellule des outils) ;
- d) le taux de $x \mapsto x^x$ entre 1 et $1 + 10^{-6}$, une fonction que tu ne sais pas encore dériver.

Puis, dans ta copie : vers quel nombre les taux de a) semblent-ils aller, et quelle dérivée de la fiche le confirme ? Quelle valeur de la dérivée de $x^x$ en 1 devines-tu avec d) ? Vérifie-la par la règle de la chaîne, en écrivant $x^x = e^{x \ln x}$ (fiche §7).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R7")  # reload your saved file

In [ ]:
with checked("R7.17a"):
    wbm.check("R7.17a", [mymath.deriv.difference_quotient(math.sqrt, 4, h) for h in (1, 0.1, 0.01)], computed=True)
with checked("R7.17b"):
    wbm.check("R7.17b", mymath.deriv.difference_quotient(math.sqrt, 4, -0.01), computed=True)
with checked("R7.17c"):
    wbm.check("R7.17c", raised(mymath.deriv.difference_quotient, math.sqrt, 4, 0), computed=True)
with checked("R7.17d"):
    wbm.check("R7.17d", mymath.deriv.difference_quotient(lambda x: x ** x, 1, 1e-6), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r7_deriv.py", "test_difference_quotient_ or test_deriv_uses")

### Ex R7.18 — La tangente à partir de f et de sa dérivée · `tangent_line` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer l'équation de la tangente à partir d'une fonction et de sa dérivée, reçues en arguments.  
**Prérequis :** fiche §4 · **mymath :** `deriv.py` · **Parcours :** C

Écris `tangent_line(f, df, a)` : le couple `(m, c)` tel que la tangente à la courbe de `f` au point d'abscisse `a` soit $y = mx + c$, `df` étant la dérivée de `f` (fiche §4 : développe $y = f(a) + f'(a)(x - a)$). Quand `f`, `df` et `a` donnent des entiers, le résultat doit rester en entiers : pas de `float(...)` ni de division.

La vérification calcule :
- a) la tangente de ln en $e$ (`math.e`), une liste `[m, c]` ;
- b) la tangente de $x \mapsto x^3 - x$ en 2, de dérivée $3x^2 - 1$ ;
- c) les types des deux nombres de b), en texte (`"int int"` si ce sont deux entiers) ;
- d) la tangente de exp en −1.

Puis, dans ta copie : où la tangente de d) coupe-t-elle l'axe des abscisses ? Recommence pour la tangente de exp en un autre point $a$ : que remarques-tu ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R7")  # reload your saved file

In [ ]:
with checked("R7.18a"):
    wbm.check("R7.18a", list(mymath.deriv.tangent_line(math.log, lambda x: 1 / x, math.e)), computed=True)
with checked("R7.18b"):
    wbm.check("R7.18b", list(mymath.deriv.tangent_line(lambda x: x ** 3 - x, lambda x: 3 * x ** 2 - 1, 2)),
              computed=True)
with checked("R7.18c"):
    result = mymath.deriv.tangent_line(lambda x: x ** 3 - x, lambda x: 3 * x ** 2 - 1, 2)
    wbm.check("R7.18c", " ".join(type(value).__name__ for value in result), computed=True)
with checked("R7.18d"):
    wbm.check("R7.18d", list(mymath.deriv.tangent_line(math.exp, math.exp, -1)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r7_deriv.py", "test_tangent_line_ or test_deriv_uses")

### Ex R7.19 — Trouver un zéro par dichotomie · `bisection` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la dichotomie, avec ses cas limites, et compter les étapes qu'elle demande.  
**Prérequis :** fiche §2 ; R6 §4 · **mymath :** `deriv.py` · **Parcours :** C, E

Écris `bisection(f, a, b, tol=1e-10, max_iter=200)` (fiche §2) : tant que l'intervalle n'est pas plus court que `tol` (et au plus `max_iter` fois), coupe-le en deux et garde la moitié où le signe de `f` change ; renvoie le milieu du dernier intervalle. Si `f` vaut exactement 0 en `a`, en `b` ou en un milieu, renvoie ce point tout de suite ; la docstring dit aussi quand lever une `ValueError`. Dans la vérification, `counted(f)` (cellule des outils) arrête une boucle qui ne finirait jamais.

La vérification calcule :
- a) le zéro de $e^x - 3x$ sur $[0 \;;\; 1]$ ;
- b) le zéro de $\ln x + x - 2$ sur $[1 \;;\; 2]$ ;
- c) le nom de l'erreur levée pour $x^2 + 1$ sur $[-1 \;;\; 1]$, puis d) pour $a = 2$ et $b = 1$ ;
- e) le résultat pour $x - 1$ sur $[1 \;;\; 3]$ ;
- f) si le zéro de $x^2 - 3$ sur $[0 \;;\; 2]$, trouvé avec `tol=1e-3`, est à moins de $10^{-3}$ de $\sqrt{3}$.

Puis g), à la main (fiche §2) : partant d'un intervalle de longueur 1, combien de coupes faut-il pour qu'il mesure moins de $10^{-10}$, la valeur par défaut de `tol` ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R7")  # reload your saved file

In [ ]:
with checked("R7.19a"):
    wbm.check("R7.19a", mymath.deriv.bisection(counted(lambda x: math.exp(x) - 3 * x), 0, 1), computed=True)
with checked("R7.19b"):
    wbm.check("R7.19b", mymath.deriv.bisection(counted(lambda x: math.log(x) + x - 2), 1, 2), computed=True)
with checked("R7.19c"):
    wbm.check("R7.19c", raised(mymath.deriv.bisection, counted(lambda x: x ** 2 + 1), -1, 1), computed=True)
with checked("R7.19d"):
    wbm.check("R7.19d", raised(mymath.deriv.bisection, counted(lambda x: x), 2, 1), computed=True)
with checked("R7.19e"):
    wbm.check("R7.19e", mymath.deriv.bisection(counted(lambda x: x - 1), 1, 3), computed=True)
with checked("R7.19f"):
    root = mymath.deriv.bisection(counted(lambda x: x ** 2 - 3), 0, 2, tol=1e-3)
    wbm.check("R7.19f", bool(abs(root - math.sqrt(3)) < 1e-3), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r7_deriv.py", "test_bisection_ or test_deriv_uses")

**g)** Le nombre de coupes, calculé à la main.

In [ ]:
answer_R7_19g = ...  # g) an integer

In [ ]:
wbm.check("R7.19g", answer_R7_19g)

## Partie B · SymPy, une prédiction, un zoom et une figure

Fiche §2 à §8. Dériver avec SymPy, trois taux d'accroissement dont il faut prévoir la limite, une courbe qui se confond avec sa tangente, puis f, f′ et des tangentes en image.

### Ex R7.20 — Vérifier ses dérivées avec `sympy.diff` 📦 ★ ⏱️ 15 min
**Objectif :** calculer des dérivées, leurs zéros et une limite avec SymPy, et tester si deux dérivées sont égales.  
**Prérequis :** fiche §5 à §7 ; R0 (SymPy) · **Parcours :** C, E

SymPy (R0) dérive exactement : `sp.diff(expression, x)`. Deux écritures d'une même dérivée peuvent différer : pour tester si deux expressions sont égales, on regarde si leur différence **se simplifie en 0** (`sp.simplify(a - b) == 0`). `sp.solve(expression, x)` donne les solutions de `expression = 0`, et `sp.limit(expression, x, 0)` la limite quand $x$ tend vers 0.

- a) La dérivée de $(x^2 - 3)\,e^{x}$, calculée par SymPy (`sp.exp`).
- b) La liste des points où cette dérivée s'annule, dans l'ordre croissant (des entiers Python : `int(...)`).
- c) La liste `proposals` (cellule ci-dessous) contient trois fonctions, chacune avec une dérivée proposée par un élève. Pour chacune, la dérivée proposée est-elle juste ? Une liste de trois booléens, dans l'ordre.
- d) $\lim_{x \to 0} \frac{\sqrt{x + 4} - 2}{x}$, avec `sp.limit` (`sp.sqrt`). De quel nombre dérivé de la fiche est-ce la limite ?

In [ ]:
x = sp.symbols("x")
proposals = [  # (function, a derivative proposed by a student)
    (x / (x + 2), 2 / (x + 2) ** 2),
    ((x ** 3 + 1) ** 2, 6 * x ** 2 * (x ** 3 + 1)),
    (sp.sqrt(x) * sp.exp(2 * x), sp.exp(2 * x) / (2 * sp.sqrt(x)) + sp.sqrt(x) * sp.exp(2 * x)),
]

In [ ]:
answer_R7_20a = ...  # a) a SymPy expression
answer_R7_20b = ...  # b) a list of integers, in increasing order
answer_R7_20c = ...  # c) a list of 3 booleans
answer_R7_20d = ...  # d) a SymPy number

In [ ]:
for letter, answer in zip("abcd", [answer_R7_20a, answer_R7_20b, answer_R7_20c, answer_R7_20d]):
    wbm.check(f"R7.20{letter}", answer)

### Ex R7.21 — Prédire : le taux d'accroissement a-t-il toujours une limite quand h tend vers 0 ? 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir, pour une fonction avec un coin, une avec un saut et une avec un raccord lisse, si le taux d'accroissement a une limite des deux côtés.  
**Prérequis :** fiche §2 et §3 · **Parcours :** C

Pour chacune des trois fonctions ci-dessous, l'expérience calcule le taux d'accroissement $\frac{f(a + h) - f(a)}{h}$ au point $a$ indiqué, pour $h = 10^{-1}, 10^{-2}, \dots, 10^{-6}$ (donc $h > 0$), puis pour les mêmes $h$ précédés d'un signe moins ($h < 0$). **Avant d'exécuter quoi que ce soit**, prédis :
- a) pour $f(x) = e^{\lvert x \rvert}$ en $a = 0$, les taux : 1 tendent vers 0 des deux côtés ; 2 tendent vers 1 quand $h > 0$ et vers −1 quand $h < 0$ ; 3 deviennent de plus en plus grands ;
- b) pour la partie entière $f(x) = \lfloor x \rfloor$ (`math.floor`, fiche §2) en $a = 2$, les taux **quand $h < 0$** : 1 tendent vers 0 ; 2 tendent vers 1 ; 3 deviennent de plus en plus grands ;
- c) pour $f(x) = \big(\max(0, x)\big)^2$ en $a = 0$, les taux : 1 tendent vers 0 des deux côtés ; 2 tendent vers 1 quand $h > 0$ et vers 0 quand $h < 0$ ; 3 n'ont de limite d'aucun côté.

Écris tes trois prédictions, vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R7_21a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_R7_21b = ...  # b) 1, 2 or 3
prediction_R7_21c = ...  # c) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abc", [prediction_R7_21a, prediction_R7_21b, prediction_R7_21c]):
    wbm.check(f"R7.21{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R7_21a, prediction_R7_21b, prediction_R7_21c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    functions = [("a) e^|x| en 0", lambda x: math.exp(abs(x)), 0),
                 ("b) partie entière en 2", math.floor, 2),
                 ("c) max(0, x)² en 0", lambda x: max(0, x) ** 2, 0)]
    for name, f, a in functions:
        print(name)
        for sign, side in ((1, "h > 0"), (-1, "h < 0")):
            steps = [sign * 10.0 ** -k for k in range(1, 7)]
            rates = [(f(a + h) - f(a)) / h for h in steps]
            print(f"   {side} :", "   ".join(f"{rate + 0.0:.6g}" for rate in rates))   # + 0.0: no "-0"

**Après l'expérience**, dans ta copie : laquelle des trois fonctions est dérivable en son point, et que vaut sa dérivée ? Pour la partie entière, que donnent les taux quand $h > 0$, et pourquoi un saut empêche-t-il toute dérivée ? Dessine l'allure des trois courbes près de leur point.

### Ex R7.22 — Zoomer sur une courbe jusqu'à voir une droite 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'écart entre une courbe et sa tangente sur des fenêtres de plus en plus étroites, et voir ce qu'un coin change.  
**Prérequis :** fiche §3 et §4 ; workbookIA 0A §100.8.3 (calcul vectorisé) · **Parcours :** C, E

Écris `max_gap(f, line, a, w)`, qui renvoie le plus grand écart $\lvert f(x) - \text{line}(x) \rvert$ pour $x$ parcourant 1 001 points également espacés de $[a - w \;;\; a + w]$ (bornes comprises : `np.linspace`). `f` et `line` reçoivent un tableau NumPy et renvoient un tableau.

La vérification calcule, pour les quatre largeurs $w$ de `widths` (0,5 ; 0,1 ; 0,01 ; 0,001) :
- a) l'écart entre ln et sa tangente en 3, $y = \frac{x}{3} + \ln 3 - 1$, divisé par $w$ (une liste de quatre nombres) ;
- b) le même écart, divisé par $w^2$ ;
- c) l'écart entre $\lvert x \rvert$ et la droite $y = 0$ autour de 0, divisé par $w$.

Puis :
- d) vers quel nombre semblent tendre les rapports de b) ? (un nombre, 3 décimales)

Dans ta copie : que disent a) et b) de la vitesse à laquelle la courbe de ln se confond avec sa tangente ? Et c) : y a-t-il une droite qui « colle » à $\lvert x \rvert$ quand on zoome sur 0 ?

In [ ]:
widths = np.array([0.5, 0.1, 0.01, 0.001])


def tangent22(x):
    """The tangent of ln at 3: y = x/3 + ln 3 − 1."""
    return x / 3 + math.log(3) - 1

In [ ]:
def max_gap(f, line, a, w):
    """The largest |f(x) − line(x)| for 1001 evenly spaced x in [a − w, a + w] (ends included)."""
    raise NotImplementedError("calcule l'écart sur la grille de np.linspace, puis son maximum")


answer_R7_22d = ...  # d) a number (3 decimals)

In [ ]:
with checked("R7.22a"):
    wbm.check("R7.22a", [max_gap(np.log, tangent22, 3, w) / w for w in widths], computed=True)
with checked("R7.22b"):
    wbm.check("R7.22b", [max_gap(np.log, tangent22, 3, w) / w ** 2 for w in widths], computed=True)
with checked("R7.22c"):
    wbm.check("R7.22c", [max_gap(np.abs, lambda x: 0 * x, 0, w) / w for w in widths], computed=True)
wbm.check("R7.22d", answer_R7_22d)

### Ex R7.23 — Tracer f, f′ et des tangentes 🎨 ★★ ⏱️ 20 min
**Objectif :** tracer une fonction avec des tangentes et sa dérivée, et relier le signe de f′ aux variations de f.  
**Prérequis :** fiche §4 et §8 ; R0 (matplotlib) · **Parcours :** C

Pour $f(x) = (x^2 - 1)^3$ (fournie : `f23`, et sa dérivée `df23`, que tu peux vérifier à la main), complète `plot_f_and_tangents(axes, f, df, points)`, qui reçoit **deux axes** côte à côte :
- dans `axes[0]` : la courbe de `f` sur $[-1{,}6 \;;\; 1{,}6]$ (au moins 20 points), et, pour chaque abscisse $p$ de `points`, la tangente en $p$ tracée sur $[p - 0{,}3 \;;\; p + 0{,}3]$, avec le point de contact ;
- dans `axes[1]` : la courbe de `df` sur le même intervalle, et la droite $y = 0$ (`axes[1].axhline(0)`) ;
- un titre pour chacun.

La cellule suivante trace la figure pour `points23 = [-1.5, -0.5, 1]`, puis une autre la vérifie. Ensuite, en lisant tes graphiques **et** en le justifiant par le calcul :
- a) les abscisses où la tangente à la courbe de $f$ est horizontale, dans l'ordre croissant ;
- b) le nombre de minimums locaux de $f$.

Dans ta copie : sur le graphique de droite, où lis-tu que $f$ monte ? La tangente en 1 est horizontale : $f$ a-t-elle pour autant un extremum en 1 ? Que fait la courbe de part et d'autre de sa tangente, et comment le lis-tu sur le graphique de droite ?

In [ ]:
def f23(x):
    return (x ** 2 - 1) ** 3


def df23(x):
    return 6 * x * (x ** 2 - 1) ** 2


points23 = [-1.5, -0.5, 1]

In [ ]:
def plot_f_and_tangents(axes, f, df, points):
    """axes[0]: f on [-1.6, 1.6] and its tangent at each p of points (on [p − 0.3, p + 0.3]); axes[1]: df and y = 0."""
    raise NotImplementedError("trace f et ses tangentes dans axes[0], f′ et y = 0 dans axes[1]")


answer_R7_23a = ...  # a) a list, in increasing order
answer_R7_23b = ...  # b) an integer

In [ ]:
with checked("R7.23 figure"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    try:
        created = drawn(plot_f_and_tangents, axes, f23, df23, points23)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex R7.23 : ta fonction ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans les axes "
              "qu'elle reçoit.")
    else:
        fig.tight_layout()
        plt.show()
for letter, answer in zip("ab", [answer_R7_23a, answer_R7_23b]):
    wbm.check(f"R7.23{letter}", answer)

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R7.23 vérifications"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    try:
        created = drawn(plot_f_and_tangents, axes, f23, df23, points23)
    finally:
        plt.close(fig)                                     # only the checks below are shown here
    problems = []
    if created:
        problems.append("ta fonction ouvre sa propre figure : dessine dans les axes reçus (axes[0].plot, …)")
    else:
        lines0 = [(np.asarray(l.get_xdata(), dtype=float), np.asarray(l.get_ydata(), dtype=float))
                  for l in axes[0].lines]
        lines1 = [(np.asarray(l.get_xdata(), dtype=float), np.asarray(l.get_ydata(), dtype=float))
                  for l in axes[1].lines]
        if not any(len(xs) >= 20 and np.allclose(ys, f23(xs)) for xs, ys in lines0):
            problems.append("la courbe de f n'est pas tracée dans axes[0] (au moins 20 points)")
        for p in points23:
            if not any(len(xs) >= 2 and np.allclose(ys, f23(p) + df23(p) * (xs - p)) and xs.min() < p < xs.max()
                       for xs, ys in lines0):
                problems.append(f"pas de tangente en {p} dans axes[0] : la droite y = f(p) + f′(p)(x − p), "
                                f"tracée avec plot de part et d'autre de p (pas axline)")
        if not any(len(xs) >= 20 and np.allclose(ys, df23(xs)) for xs, ys in lines1):
            problems.append("la courbe de f′ n'est pas tracée dans axes[1] (au moins 20 points)")
        if not any(len(ys) >= 2 and np.allclose(ys, 0) for xs, ys in lines1):
            problems.append("la droite y = 0 dans axes[1] (axes[1].axhline(0))")
        if not all(ax.get_title() for ax in axes):
            problems.append("un titre pour chaque graphique (ax.set_title)")
    if problems:
        print("❌ Ex R7.23 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R7.23 : f, ses trois tangentes et f′, avec la droite y = 0.")

## Partie C · Dérivées fausses et tests

Fiche §5 à §7. Quatre dérivées fausses à trouver, puis un test qui vérifie une dérivée.

### Ex R7.24 — Chasse à l'erreur : quatre dérivées fausses 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer des dérivées fausses dans du code (règle du produit, règle de la chaîne), les corriger et les vérifier par un taux d'accroissement.  
**Prérequis :** fiche §6 et §7 · **Parcours :** C, E

Six fonctions Python calculent les dérivées de six fonctions (écrites en commentaire). Certaines lignes sont fausses. **Sans rien exécuter d'abord**, trouve-les.

```python
def d_a(x):  # derivative of x**2 * sqrt(x + 1)
    return 2 * x * (1 / (2 * math.sqrt(x + 1)))                   # (A)


def d_b(x):  # derivative of (x + 2) / (x - 3)
    return -5 / (x - 3) ** 2                                      # (B)


def d_c(x):  # derivative of exp(3 * x)
    return math.exp(3 * x)                                        # (C)


def d_d(x):  # derivative of ln(x**3 + 1)
    return 1 / (x ** 3 + 1)                                       # (D)


def d_e(x):  # derivative of sqrt(2 * x)
    return 1 / (2 * math.sqrt(2 * x))                             # (E)


def d_f(x):  # derivative of x**3 - 1 / x
    return 3 * x ** 2 + 1 / x ** 2                                # (F)
```

- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;

Puis corrige-les dans la cellule ci-dessous (elle contient le code fautif). La fonction `slope(f, x)` de la cellule fournie calcule un taux d'accroissement avec $h = 10^{-6}$ : compare-le à chacune de tes dérivées en $x = 2$ pour les vérifier. La vérification calcule b), la liste des six valeurs `[d_a(2), d_b(2), …, d_f(2)]`.

Dans ta copie : pour chaque ligne fausse, quelle règle a été oubliée ? Laquelle des deux lignes justes pouvait sembler fausse, et pourquoi ne l'est-elle pas ?

In [ ]:
def slope(f, x, h=1e-6):
    """A rate of change of f between x and x + h: close to f'(x) for a small h (fiche §3)."""
    return (f(x + h) - f(x)) / h

In [ ]:
answer_R7_24a = ...  # a) letters, as text


# correct the six functions below (the wrong lines only)
def d_a(x):  # derivative of x**2 * sqrt(x + 1)
    return 2 * x * (1 / (2 * math.sqrt(x + 1)))                   # (A)


def d_b(x):  # derivative of (x + 2) / (x - 3)
    return -5 / (x - 3) ** 2                                      # (B)


def d_c(x):  # derivative of exp(3 * x)
    return math.exp(3 * x)                                        # (C)


def d_d(x):  # derivative of ln(x**3 + 1)
    return 1 / (x ** 3 + 1)                                       # (D)


def d_e(x):  # derivative of sqrt(2 * x)
    return 1 / (2 * math.sqrt(2 * x))                             # (E)


def d_f(x):  # derivative of x**3 - 1 / x
    return 3 * x ** 2 + 1 / x ** 2                                # (F)

In [ ]:
wbm.check("R7.24a", answer_R7_24a)
if answer_R7_24a is ...:
    print("⏳ Ex R7.24b : trouve d'abord les lignes fausses (a), puis corrige les fonctions.")
else:
    with checked("R7.24b"):
        wbm.check("R7.24b", [d_a(2), d_b(2), d_c(2), d_d(2), d_e(2), d_f(2)], computed=True)

### Ex R7.25 — Tester une dérivée avec pytest et SymPy 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test qui compare une dérivée calculée à la main à celle de SymPy, et vérifier qu'il sait dire non.  
**Prérequis :** fiche §5 et §6 ; R0 (pytest) · **Parcours :** C

Une dérivée calculée à la main se vérifie par un **test** : une fonction `test_...` que pytest exécute, et qui échoue si la dérivée est fausse. Ici :
1. Écris `dg(x)`, la dérivée de $g(x) = \frac{x^2 + 1}{x + 3}$, **calculée à la main** (règle du quotient, simplifiée) ;
2. Écris `test_dg_matches_sympy()`, **sans argument** : il crée son propre symbole (`X = sp.symbols("x")`, **dans** la fonction), calcule la dérivée de $g$ avec SymPy (`sp.diff`), puis vérifie avec `assert` et `math.isclose` que `dg` donne la même valeur qu'elle en au moins quatre points (pense à un point négatif ; évite −3). Pour évaluer une expression SymPy en un point : `float(expression.subs(X, point))`. Un message après l'`assert` (`assert ..., f"en {point} : {dg(point)}"`) dira où ça coince.

pytest exécute ta fonction de test **seule**, sans les autres cellules du notebook : tout ce qu'elle utilise (son symbole, l'expression de $g$) doit être défini dedans ; seuls `math`, `sp` et `dg` lui sont fournis.

La vérification lance ton test avec pytest (`wbm.run_pytest`) deux fois : sur ta `dg`, où il doit passer ; puis sur `dg_bug` (fournie), une dérivée fausse, où il doit **échouer**. Un test qui ne sait pas dire non ne vérifie rien.
- a) ton test passe-t-il sur `dg` ? ; b) échoue-t-il sur `dg_bug` ?

In [ ]:
def dg_bug(x):
    """A wrong derivative of g(x) = (x**2 + 1) / (x + 3): u'v + uv' over v², with a + instead of a −."""
    return (2 * x * (x + 3) + (x ** 2 + 1)) / (x + 3) ** 2

In [ ]:
def dg(x):
    """The derivative of g(x) = (x**2 + 1) / (x + 3), computed by hand (x != -3)."""
    raise NotImplementedError("écris la dérivée de g, calculée à la main")


def test_dg_matches_sympy():
    """dg agrees with the derivative of g computed by SymPy, at four points or more."""
    raise NotImplementedError("écris le test : compare dg à sp.diff en quelques points")

In [ ]:
with checked("R7.25"):
    dg(1.0)                        # ⏳ while dg is not written
    test_dg_matches_sympy()        # your test runs here first: ⏳ while it is not written
    good = wbm.run_pytest(test_dg_matches_sympy, subject=dg, preamble="import sympy as sp", quiet=True)
    bad = wbm.run_pytest(test_dg_matches_sympy, subject=dg_bug, name="dg", preamble="import sympy as sp", quiet=True)
    print(f"sur dg : {good.summary} ; sur dg_bug : {bad.summary}")
    if "NameError" in good.output + bad.output:
        print("⚠️ Ton test utilise un nom défini hors de lui (NameError) : pytest l'exécute seul, sans les autres "
              "cellules. Définis dedans tout ce qu'il utilise (son symbole X, l'expression de g).")
    wbm.check("R7.25a", good.ok, computed=True)
    wbm.check("R7.25b", bad.failed > 0 and "NameError" not in bad.output, computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu une limite en l'infini ou en un point, et repères-tu une asymptote ?
2. Dérives-tu sans hésiter un produit, un quotient et une composée, et écris-tu une tangente ?
3. Sais-tu étudier les variations d'une fonction et trouver son minimum, y compris aux bords ?

**La suite** : R8, *Probabilités de base*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.